[PyArrow](https://arrow.apache.org/docs/python/) is the Python implementation of [Apache Arrow](https://arrow.apache.org/), providing efficient columnar data structures and compute functions for data analysis. Unlike row-oriented tools, PyArrow operates directly on columnar data, enabling fast vectorized operations without converting to other formats.

This notebook demonstrates exploratory data analysis (EDA) on the [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) dataset using PyArrow's compute module. You'll learn how to inspect schemas, analyze missing data, compute descriptive statistics, perform group-by aggregations, and filter data using PyArrow expressions.

Data is loaded via [ADBC](https://arrow.apache.org/adbc/) (Arrow Database Connectivity), which retrieves query results directly as Arrow tables. This notebook uses [DuckDB](https://duckdb.org/) for simplicity, but ADBC supports a wide range of SQL databases including Snowflake, Databricks, BigQuery, PostgreSQL, MySQL, SQLite, and more.

## Setup

Install the required dependencies. `adbc-driver-manager` provides the core ADBC interface, and `dbc` is a CLI for managing ADBC driver installations:

In [5]:
%pip install -q dbc adbc-driver-manager pyarrow

Note: you may need to restart the kernel to use updated packages.


Use `dbc` to install the DuckDB ADBC driver:

In [ ]:
!dbc install -q duckdb

Import PyArrow and the ADBC driver manager:

In [7]:
import pyarrow.compute as pc
from adbc_driver_manager import dbapi

## Load Data

Load the Palmer Penguins dataset using ADBC. The result is returned directly as an Arrow table, so no conversion from rows to columns is needed:

In [ ]:
with (
    dbapi.connect(driver="duckdb") as connection,
    connection.cursor() as cursor,
):
    cursor.execute("FROM read_csv('https://blobs.duckdb.org/data/penguins.csv', nullstr = 'NA')")
    table = cursor.fetch_arrow_table()

## Dataset Overview

Inspect the table schema to see column names and data types:

In [9]:
table.schema

species: string
island: string
bill_length_mm: double
bill_depth_mm: double
flipper_length_mm: int64
body_mass_g: int64
sex: string
year: int64

Check the number of rows and columns:

In [10]:
table.shape

(344, 8)

Preview the first five rows:

In [11]:
table.slice(0, 5)

pyarrow.Table
species: string
island: string
bill_length_mm: double
bill_depth_mm: double
flipper_length_mm: int64
body_mass_g: int64
sex: string
year: int64
----
species: [["Adelie","Adelie","Adelie","Adelie","Adelie"]]
island: [["Torgersen","Torgersen","Torgersen","Torgersen","Torgersen"]]
bill_length_mm: [[39.1,39.5,40.3,null,36.7]]
bill_depth_mm: [[18.7,17.4,18,null,19.3]]
flipper_length_mm: [[181,186,195,null,193]]
body_mass_g: [[3750,3800,3250,null,3450]]
sex: [["male","female","female",null,"female"]]
year: [[2007,2007,2007,2007,2007]]

## Missing Data

Count null values in each column:

In [12]:
{column: table.column(column).null_count for column in table.column_names}

{'species': 0,
 'island': 0,
 'bill_length_mm': 2,
 'bill_depth_mm': 2,
 'flipper_length_mm': 2,
 'body_mass_g': 2,
 'sex': 11,
 'year': 0}

Calculate the percentage of missing values per column:

In [13]:
{column: table.column(column).null_count / table.num_rows * 100 for column in table.column_names}

{'species': 0.0,
 'island': 0.0,
 'bill_length_mm': 0.5813953488372093,
 'bill_depth_mm': 0.5813953488372093,
 'flipper_length_mm': 0.5813953488372093,
 'body_mass_g': 0.5813953488372093,
 'sex': 3.1976744186046515,
 'year': 0.0}

## Descriptive Statistics

Compute statistics for numeric columns:

In [14]:
{
    column_name: {
        "count": pc.count(column).as_py(),
        "min": pc.min(column).as_py(),
        "max": pc.max(column).as_py(),
        "mean": pc.mean(column).as_py(),
        "stddev": pc.stddev(column).as_py(),
    }
    for column_name in [
        "bill_length_mm",
        "bill_depth_mm",
        "flipper_length_mm",
        "body_mass_g",
        "year",
    ]
    for column in [table.column(column_name)]
}

{'bill_length_mm': {'count': 342,
  'min': 32.1,
  'max': 59.6,
  'mean': 43.92192982456141,
  'stddev': 5.4515960231618195},
 'bill_depth_mm': {'count': 342,
  'min': 13.1,
  'max': 21.5,
  'mean': 17.151169590643274,
  'stddev': 1.9719039187562526},
 'flipper_length_mm': {'count': 342,
  'min': 172,
  'max': 231,
  'mean': 200.91520467836258,
  'stddev': 14.041140568589102},
 'body_mass_g': {'count': 342,
  'min': 2700,
  'max': 6300,
  'mean': 4201.754385964912,
  'stddev': 800.781229238452},
 'year': {'count': 344,
  'min': 2007,
  'max': 2009,
  'mean': 2008.0290697674418,
  'stddev': 0.8171655889620333}}

Compute value counts for categorical columns:

In [15]:
{
    column: {
        item["values"].as_py(): item["counts"].as_py()
        for item in pc.value_counts(table.column(column))
    }
    for column in ["species", "island", "sex"]
}

{'species': {'Adelie': 152, 'Gentoo': 124, 'Chinstrap': 68},
 'island': {'Torgersen': 52, 'Biscoe': 168, 'Dream': 124},
 'sex': {'male': 168, 'female': 165, None: 11}}

## Group-by Aggregations

Count penguins by island:

In [16]:
table.group_by("island").aggregate([("species", "count")]).to_pylist()

[{'island': 'Torgersen', 'species_count': 52},
 {'island': 'Biscoe', 'species_count': 168},
 {'island': 'Dream', 'species_count': 124}]

Calculate mean measurements grouped by species:

In [17]:
table.group_by("species").aggregate(
    [
        ("bill_length_mm", "mean"),
        ("bill_depth_mm", "mean"),
        ("flipper_length_mm", "mean"),
        ("body_mass_g", "mean"),
    ]
).to_pylist()

[{'species': 'Adelie',
  'bill_length_mm_mean': 38.79139072847684,
  'bill_depth_mm_mean': 18.346357615894032,
  'flipper_length_mm_mean': 189.95364238410596,
  'body_mass_g_mean': 3700.662251655629},
 {'species': 'Gentoo',
  'bill_length_mm_mean': 47.504878048780476,
  'bill_depth_mm_mean': 14.982113821138206,
  'flipper_length_mm_mean': 217.1869918699187,
  'body_mass_g_mean': 5076.016260162602},
 {'species': 'Chinstrap',
  'bill_length_mm_mean': 48.83382352941177,
  'bill_depth_mm_mean': 18.420588235294115,
  'flipper_length_mm_mean': 195.8235294117647,
  'body_mass_g_mean': 3733.0882352941176}]

## Data Filtering

Filter penguins by species:

In [18]:
gentoo = table.filter(pc.equal(table.column("species"), "Gentoo"))
gentoo.slice(0, 5)

pyarrow.Table
species: string
island: string
bill_length_mm: double
bill_depth_mm: double
flipper_length_mm: int64
body_mass_g: int64
sex: string
year: int64
----
species: [["Gentoo","Gentoo","Gentoo","Gentoo","Gentoo"]]
island: [["Biscoe","Biscoe","Biscoe","Biscoe","Biscoe"]]
bill_length_mm: [[46.1,50,48.7,50,47.6]]
bill_depth_mm: [[13.2,16.3,14.1,15.2,14.5]]
flipper_length_mm: [[211,230,210,218,215]]
body_mass_g: [[4500,5700,4450,5700,5400]]
sex: [["female","male","female","male","male"]]
year: [[2007,2007,2007,2007,2007]]

Filter by numeric condition (body mass greater than 5000g):

In [19]:
heavy_penguins = table.filter(pc.greater(table.column("body_mass_g"), 5000))
print(f"Penguins over 5000g: {heavy_penguins.num_rows}")
heavy_penguins.slice(0, 5)

Penguins over 5000g: 61


pyarrow.Table
species: string
island: string
bill_length_mm: double
bill_depth_mm: double
flipper_length_mm: int64
body_mass_g: int64
sex: string
year: int64
----
species: [["Gentoo","Gentoo","Gentoo","Gentoo","Gentoo"]]
island: [["Biscoe","Biscoe","Biscoe","Biscoe","Biscoe"]]
bill_length_mm: [[50,50,47.6,46.7,46.8]]
bill_depth_mm: [[16.3,15.2,14.5,15.3,15.4]]
flipper_length_mm: [[230,218,215,219,215]]
body_mass_g: [[5700,5700,5400,5200,5150]]
sex: [["male","male","male","male","male"]]
year: [[2007,2007,2007,2007,2007]]

Combine multiple filter conditions (Adelie penguins on Dream island):

In [20]:
is_adelie = pc.equal(table.column("species"), "Adelie")
is_dream = pc.equal(table.column("island"), "Dream")
adelie_dream = table.filter(pc.and_(is_adelie, is_dream))
print(f"Adelie penguins on Dream island: {adelie_dream.num_rows}")
adelie_dream.slice(0, 3)

Adelie penguins on Dream island: 56


pyarrow.Table
species: string
island: string
bill_length_mm: double
bill_depth_mm: double
flipper_length_mm: int64
body_mass_g: int64
sex: string
year: int64
----
species: [["Adelie","Adelie","Adelie"]]
island: [["Dream","Dream","Dream"]]
bill_length_mm: [[39.5,37.2,39.5]]
bill_depth_mm: [[16.7,18.1,17.8]]
flipper_length_mm: [[178,178,188]]
body_mass_g: [[3250,3900,3300]]
sex: [["female","male","female"]]
year: [[2007,2007,2007]]